# Review Collection from Steam

Collects the early-access and post-launch review windows for every candidate game. Resumable, checkpointed per game.

# Setup and Imports

In [14]:
import requests
import os
import json
import time
import random
import threading
import datetime as dt
from concurrent.futures import ThreadPoolExecutor,as_completed
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# paths in one place
project_root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
raw_reviews_dir=project_root/'data'/'raw'/'reviews'
checkpoint_dir=project_root/'data'/'raw'/'_checkpoints'
meta_dir=project_root/'data'/'raw'/'meta'
for d in (raw_reviews_dir,checkpoint_dir,meta_dir):
    d.mkdir(parents=True,exist_ok=True)

print('project root:',project_root)
print('reviews go to:',raw_reviews_dir)
print('checkpoints go to:',checkpoint_dir)

project root: d:\Msc_Data_Analytics\Research_Practicum_final\final_thesis
reviews go to: d:\Msc_Data_Analytics\Research_Practicum_final\final_thesis\data\raw\reviews
checkpoints go to: d:\Msc_Data_Analytics\Research_Practicum_final\final_thesis\data\raw\_checkpoints


## Collection settings

All the tuning constants in one place.

In [15]:
# steam caps a request at 100 reviews
reviews_per_page=100

# delay between requests
seconds_between_requests=0.4

# steam early access began in 2013
earliest_possible_review=dt.datetime(2012,1,1)

# outcome window, days after release
post_launch_window_days=180

# stop splitting a window below this width
minimum_window_seconds=3600

# cap per window so one huge game cannot dominate the corpus
# 5000 per game is ample for a per-game average
reviews_cap_per_window=5000

# below this size the cursor is cheaper than windows
whole_history_cursor_threshold=8000

# concurrent workers, one game each
parallel_workers=8

# a real user agent with a contact address
request_headers={'User-Agent':'EA-Health-Scoring-research/1.0 (academic study; mastudsaurabh33@gmail.com)'}

print('reviews per page:',reviews_per_page)
print('post launch window:',post_launch_window_days,'days')
print('polite delay between requests:',seconds_between_requests,'seconds')
print('parallel workers:',parallel_workers)

reviews per page: 100
post launch window: 180 days
polite delay between requests: 0.4 seconds
parallel workers: 8


# Talking to the Steam API

One session per thread, retry with backoff, and every request counted.

In [16]:
# one session per thread
thread_local_storage=threading.local()

def get_session():
    if not hasattr(thread_local_storage,'session'):
        thread_local_storage.session=requests.Session()
        thread_local_storage.session.headers.update(request_headers)
    return thread_local_storage.session

# count requests to measure throughput
request_counter={'total':0,'retries':0,'failures':0}
counter_lock=threading.Lock()

def bump(key):
    with counter_lock:
        request_counter[key]+=1


def fetch_review_page(appid,cursor='*',start_date=None,end_date=None,max_attempts=5):
    """one call to the steam appreviews endpoint. returns the parsed json or None if it failed"""
    params={'json':1,
            'language':'all',
            'num_per_page':reviews_per_page,
            'purchase_type':'all',
            'review_type':'all',
            'filter':'recent',
            'cursor':cursor}

    # a date window needs filter=all, filter=recent ignores it
    if start_date is not None:
        params['filter']='all'
        params['start_date']=int(start_date)
        params['end_date']=int(end_date)
        params['date_range_type']='include'

    url=f'https://store.steampowered.com/appreviews/{appid}'

    for attempt in range(max_attempts):
        try:
            bump('total')
            response=get_session().get(url,params=params,timeout=30)

            # 429 is rate limiting, 5xx is steam being unwell, both worth a retry
            if response.status_code==429 or response.status_code>=500:
                bump('retries')
                backoff_seconds=(2**attempt)+random.random()
                print(f'  http {response.status_code} on app {appid}, backing off {backoff_seconds:.1f}s (attempt {attempt+1})')
                time.sleep(backoff_seconds)
                continue

            response.raise_for_status()
            payload=response.json()

            if payload.get('success')!=1:
                bump('failures')
                return None

            time.sleep(seconds_between_requests)
            return payload

        except (requests.RequestException,ValueError) as problem:
            bump('retries')
            backoff_seconds=(2**attempt)+random.random()
            print(f'  request problem on app {appid}: {type(problem).__name__}, retrying in {backoff_seconds:.1f}s')
            time.sleep(backoff_seconds)

    # every attempt failed, fail loudly
    bump('failures')
    print(f'  GAVE UP on app {appid} after {max_attempts} attempts')
    return None


print('api helper is ready')

api helper is ready


# Checking the API actually gives me the fields i need

In [17]:
# rust left early access in 2018, so it is a good first check
test_appid=252490
test_payload=fetch_review_page(test_appid)
test_review=test_payload['reviews'][0]

print('WHAT ONE STEAM REVIEW ACTUALLY CONTAINS')
print('top level fields:',sorted(test_review.keys()))
print()
print('author fields:',sorted(test_review['author'].keys()))

print('THE FOUR FIELDS THIS PROJECT DEPENDS ON')
print('written_during_early_access :',test_review.get('written_during_early_access'))
print('playtime_at_review          :',test_review['author'].get('playtime_at_review'))
print('playtime_forever            :',test_review['author'].get('playtime_forever'))
print('steamid                     :',test_review['author'].get('steamid'))

# total review count, used to plan the collection
print('query summary:',json.dumps(test_payload['query_summary'],indent=2))

WHAT ONE STEAM REVIEW ACTUALLY CONTAINS
top level fields: ['app_release_date', 'author', 'comment_count', 'language', 'primarily_steam_deck', 'reactions', 'received_for_free', 'recommendationid', 'refunded', 'review', 'steam_purchase', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_funny', 'votes_up', 'weighted_vote_score', 'written_during_early_access']

author fields: ['avatar', 'last_played', 'num_games_owned', 'num_reviews', 'persona_status', 'personaname', 'playtime_at_review', 'playtime_forever', 'playtime_last_two_weeks', 'profile_url', 'steamid']
THE FOUR FIELDS THIS PROJECT DEPENDS ON
written_during_early_access : False
playtime_at_review          : 9
playtime_forever            : 9
steamid                     : 76561199030908604
query summary: {
  "num_reviews": 100,
  "review_score": 8,
  "review_score_desc": "Very Positive",
  "total_positive": 1203996,
  "total_negative": 182994,
  "total_reviews": 1386990
}


# Finding the Early Access boundary for free

The boundary is derived from the review flag, not from the store release date.

In [18]:
def get_game_summary(appid):
    """one request that tells me the release date, the total review count and whether the game is still in early access"""
    payload=fetch_review_page(appid)
    if payload is None or not payload.get('reviews'):
        return None

    first_review=payload['reviews'][0]
    summary=payload['query_summary']

    # comes back as a string, sometimes with a decimal, so go via float
    raw_release_date=first_review.get('app_release_date')
    release_timestamp=int(float(raw_release_date)) if raw_release_date else None

    # newest reviews still flagged early access means the game has not launched
    newest_reviews_are_ea=[r.get('written_during_early_access') for r in payload['reviews'][:20]]
    still_in_early_access=sum(newest_reviews_are_ea)>len(newest_reviews_are_ea)*0.5

    return {'appid':appid,
            'release_timestamp':release_timestamp,
            'release_date':dt.datetime.utcfromtimestamp(release_timestamp).date().isoformat() if release_timestamp else None,
            'total_reviews':summary.get('total_reviews'),
            'total_positive':summary.get('total_positive'),
            'total_negative':summary.get('total_negative'),
            'review_score_desc':summary.get('review_score_desc'),
            'still_in_early_access':still_in_early_access}


rust_summary=get_game_summary(252490)
print('rust summary:')
for key,value in rust_summary.items():
    print(f'  {key}: {value}')

# confirms the date lines up with rust leaving early access

rust summary:
  appid: 252490
  release_timestamp: 1518112367
  release_date: 2018-02-08
  total_reviews: 1386990
  total_positive: 1203996
  total_negative: 182994
  review_score_desc: Very Positive
  still_in_early_access: False


## The store release date is not the launch date

For many games it records when early access *started*, so the boundary has to be bisected from the flag instead.

In [19]:
def find_launch_date(appid,probe_days=30,max_probes=20):
    # binary search for where the early access flag flips
    # returns the launch timestamp, or None if still in EA
    lo=int(earliest_possible_review.timestamp())
    hi=int(time.time())
    half=probe_days*86400//2
    probes_used=0

    def early_access_share(at_time):
        payload=fetch_review_page(appid,start_date=max(at_time-half,lo),end_date=min(at_time+half,hi))
        rows=payload.get('reviews',[]) if payload else []
        # an edited review carries the flag from its edit date
        untouched=[r for r in rows if int(r['timestamp_updated'])==int(r['timestamp_created'])]
        judge_on=untouched or rows
        if not judge_on:
            return None
        return sum(bool(r['written_during_early_access']) for r in judge_on)/len(judge_on)

    # still flagged at the newest review means not launched
    newest_share=early_access_share(hi-half)
    probes_used+=1
    if newest_share is not None and newest_share>0.5:
        return None

    while hi-lo>86400 and probes_used<max_probes:
        midpoint=(lo+hi)//2
        share=early_access_share(midpoint)
        probes_used+=1
        if share is None or share>0.5:
            lo=midpoint
        else:
            hi=midpoint

    return {'launch_timestamp':hi,
            'launch_date':dt.datetime.utcfromtimestamp(hi).date().isoformat(),
            'bisect_probes':probes_used,
            'bisect_bracket_hours':round((hi-lo)/3600,1)}


# check against four games with known answers
print('LAUNCH DATE BISECT vs GAMES I KNOW THE ANSWER FOR')
known_launches={242760:('The Forest','2018-04-30'),
                252490:('Rust','2018-02-08'),
                1604030:('V Rising','2024-05-08'),
                632360:('Risk of Rain 2','2020-08-11')}
for check_appid,(game_name,true_launch) in known_launches.items():
    found=find_launch_date(check_appid)
    days_off=(dt.date.fromisoformat(found['launch_date'])-dt.date.fromisoformat(true_launch)).days
    print(f'{game_name:16s} bisect={found["launch_date"]}  true={true_launch}  '
          f'off by {days_off:+d}d  in {found["bisect_probes"]} probes, '
          f'bracket {found["bisect_bracket_hours"]}h')
print('within a week on all four and always a few days early, exactly as expected')

LAUNCH DATE BISECT vs GAMES I KNOW THE ANSWER FOR
The Forest       bisect=2018-04-30  true=2018-04-30  off by +0d  in 14 probes, bracket 15.6h
Rust             bisect=2018-02-05  true=2018-02-08  off by -3d  in 14 probes, bracket 15.6h
V Rising         bisect=2024-05-04  true=2024-05-08  off by -4d  in 14 probes, bracket 15.6h
Risk of Rain 2   bisect=2020-08-09  true=2020-08-11  off by -2d  in 14 probes, bracket 15.6h
within a week on all four and always a few days early, exactly as expected


# Collecting a whole time window

Splits a window when it comes back full, and caps how much any one game can contribute.

In [20]:
def collect_time_window(appid,window_start,window_end,review_file,seen_review_ids,
                        review_cap,label,progress_every=25):
    """pull every review between two timestamps, splitting windows that come back full.

    writes to review_file as it goes and returns the pending stack so a crash is resumable"""

    pending_windows=[(window_start,window_end)]
    reviews_written=0
    truncated_windows=0
    windows_done=0

    while pending_windows:
        current_start,current_end=pending_windows.pop()
        payload=fetch_review_page(appid,start_date=current_start,end_date=current_end)
        windows_done+=1

        if payload is None:
            truncated_windows+=1
            continue

        reviews=payload.get('reviews',[])

        # write straight to disk, deduplicated on recommendationid
        # writing as i go keeps a crash cheap
        for review in reviews:
            review_id=review['recommendationid']
            if review_id in seen_review_ids:
                continue
            seen_review_ids.add(review_id)
            review['_appid']=appid
            review['_window']=label
            review_file.write(json.dumps(review,ensure_ascii=False)+'\n')
            reviews_written+=1

        # a full page means more is held back, so split the window
        window_is_full=len(reviews)>=reviews_per_page
        window_width=current_end-current_start

        if window_is_full and window_width>minimum_window_seconds:
            # split into four, not two - fewer wasted parent requests
            edges=[current_start+(window_width*part)//4 for part in range(5)]
            for part in range(4):
                if edges[part+1]>edges[part]:
                    pending_windows.append((edges[part],edges[part+1]))
        elif window_is_full:
            # an hour wide and still full, accept the loss but count it
            truncated_windows+=1

        # record whenever the cap binds
        if reviews_written>=review_cap:
            print(f'  app {appid} {label}: hit the {review_cap:,} review cap, stopping this window')
            review_file.flush()
            return reviews_written,truncated_windows,True,len(pending_windows)

        if windows_done%progress_every==0:
            review_file.flush()
            print(f'  app {appid} {label}: {reviews_written:,} reviews, '
                  f'{len(pending_windows)} windows still pending',flush=True)

    review_file.flush()
    return reviews_written,truncated_windows,False,0


print('window collector is ready')

window collector is ready


# Collecting one game end to end

Routes by size: cursor for small games, time windows for large ones.

In [21]:
def checkpoint_path(appid):
    return checkpoint_dir/f'{appid}.json'

def reviews_path(appid):
    return raw_reviews_dir/f'{appid}.jsonl'

def part_path(appid):
    return raw_reviews_dir/f'{appid}.jsonl.part'


def sweep_orphan_parts():
    # a leftover .part means a dead run, so clear it
    orphans=list(raw_reviews_dir.glob('*.jsonl.part'))
    for orphan in orphans:
        orphan.unlink()
    print(f'orphan .part files cleared: {len(orphans)}')
    return len(orphans)


# stamp each checkpoint with its run id so old corpora are obvious
corpus_snapshot_id=dt.datetime.now().strftime('%Y%m%dT%H%M%S')
print('corpus snapshot id for this run:',corpus_snapshot_id)


def collect_stratified_slices(appid,window_start,window_end,review_file,seen_review_ids,
                              review_cap,label):
    # cap will bind, so sample evenly instead of walking
    # even slices keep the sample spread for the trajectory features
    number_of_slices=max(int(review_cap/reviews_per_page),1)
    slice_width=max((window_end-window_start)//number_of_slices,1)
    reviews_written=0
    empty_slices=0

    for slice_index in range(number_of_slices):
        slice_start=window_start+slice_index*slice_width
        slice_end=slice_start+slice_width if slice_index<number_of_slices-1 else window_end
        payload=fetch_review_page(appid,start_date=slice_start,end_date=slice_end)
        rows=payload.get('reviews',[]) if payload else []
        if not rows:
            empty_slices+=1
            continue
        for review in rows:
            review_id=review['recommendationid']
            if review_id in seen_review_ids:
                continue
            seen_review_ids.add(review_id)
            review['_appid']=appid
            review['_window']=label
            review_file.write(json.dumps(review,ensure_ascii=False)+'\n')
            reviews_written+=1
        if reviews_written>=review_cap:
            break
    review_file.flush()
    return reviews_written,empty_slices


def collect_whole_history_by_cursor(appid,review_file,seen_review_ids,review_cap):
    # cursor walk, which only advances without a date range
    cursor='*'
    reviews_written=0
    while True:
        payload=fetch_review_page(appid,cursor=cursor)
        rows=payload.get('reviews',[]) if payload else []
        if not rows:
            break
        for review in rows:
            review_id=review['recommendationid']
            if review_id in seen_review_ids:
                continue
            seen_review_ids.add(review_id)
            review['_appid']=appid
            review['_window']='whole_history'
            review_file.write(json.dumps(review,ensure_ascii=False)+'\n')
            reviews_written+=1
        next_cursor=payload.get('cursor') if payload else None
        if not next_cursor or next_cursor==cursor:
            break
        cursor=next_cursor
        if reviews_written>=review_cap:
            review_file.flush()
            return reviews_written,True
    review_file.flush()
    return reviews_written,False


def collect_game(appid,force=False):
    # collect both review windows for one game
    checkpoint_file=checkpoint_path(appid)

    # skip finished games, which makes the run resumable
    if checkpoint_file.exists() and not force:
        existing=json.loads(checkpoint_file.read_text())
        if existing.get('done'):
            print(f'app {appid}: already done, {existing.get("n_reviews")} reviews, skipping')
            return existing

    started_at=time.time()
    requests_before=request_counter['total']

    summary=get_game_summary(appid)
    if summary is None:
        print(f'app {appid}: no reviews returned, skipping')
        return None

    # the store date is not the launch date, so bisect the flag
    launch=find_launch_date(appid)
    still_in_early_access=launch is None

    record=dict(summary)
    record['listing_timestamp']=summary['release_timestamp']

    if still_in_early_access:
        # no launch, so no outcome window - keep the EA rows
        launch_timestamp=int(time.time())
        record.update({'cohort':'still_in_ea','derived_launch_timestamp':None,
                       'launch_date':None,'listing_vs_launch_delta_days':None,
                       'bisect_probes':None,'bisect_bracket_hours':None})
    else:
        launch_timestamp=launch['launch_timestamp']
        # the gap between the two dates is itself a finding
        delta_days=round((launch_timestamp-summary['release_timestamp'])/86400,1) \
                   if summary['release_timestamp'] else None
        record.update({'cohort':'launched',
                       'derived_launch_timestamp':launch_timestamp,
                       'launch_date':launch['launch_date'],
                       'listing_vs_launch_delta_days':delta_days,
                       'bisect_probes':launch['bisect_probes'],
                       'bisect_bracket_hours':launch['bisect_bracket_hours']})

        months_since_launch=(time.time()-launch_timestamp)/(86400*30.4)
        if months_since_launch<6:
            print(f'app {appid}: launched only {months_since_launch:.1f} months ago, too recent, skipping')
            record.update({'done':True,'excluded':'released_too_recently','n_reviews':0,
                           'n_flagged_early_access':0,
                           'requests_used':request_counter['total']-requests_before})
            record['corpus_snapshot_id']=corpus_snapshot_id
            checkpoint_file.write_text(json.dumps(record,indent=2))
            return record

    seen_review_ids=set()

    # route by the total steam reports, not by rows on disk
    # small games: cursor. large games: windows aimed at the EA period
    # because the cursor starts at today and would never reach it
    total_reviews=summary.get('total_reviews') or 0
    use_cursor=total_reviews<=whole_history_cursor_threshold
    record.setdefault('window_method','n/a')
    record['collection_strategy']='whole_history_cursor' if use_cursor else 'time_windows'

    # write to .part - opening the real file with 'w' truncates it
    # .part becomes the real file only after the checkpoint is written
    with open(part_path(appid),'w',encoding='utf-8') as review_file:

        if use_cursor:
            n_all,all_capped=collect_whole_history_by_cursor(
                appid,review_file,seen_review_ids,reviews_cap_per_window*2)
            n_ea,n_post=n_all,0
            ea_truncated=post_truncated=0
            ea_capped,post_capped=all_capped,False
            ea_pending=post_pending=0
        else:
            # cap will bind, so sample evenly rather than walk exhaustively
            cap_will_bind=total_reviews>reviews_cap_per_window*2
            record['window_method']='stratified' if cap_will_bind else 'bisect'

            ea_window_start=int(earliest_possible_review.timestamp())
            if cap_will_bind:
                n_ea,ea_empty=collect_stratified_slices(
                    appid,ea_window_start,launch_timestamp,review_file,seen_review_ids,
                    reviews_cap_per_window,'early_access')
                ea_truncated,ea_capped,ea_pending=0,True,0
            else:
                n_ea,ea_truncated,ea_capped,ea_pending=collect_time_window(
                    appid,ea_window_start,launch_timestamp,review_file,seen_review_ids,
                    reviews_cap_per_window,'early_access')

            # window 2: the first 180 days after launch
            # a game still in early access has no such window
            if still_in_early_access:
                n_post,post_truncated,post_capped,post_pending=0,0,False,0
            elif cap_will_bind:
                post_window_end=launch_timestamp+post_launch_window_days*86400
                n_post,post_empty=collect_stratified_slices(
                    appid,launch_timestamp,post_window_end,review_file,seen_review_ids,
                    reviews_cap_per_window,'post_launch')
                post_truncated,post_capped,post_pending=0,True,0
            else:
                post_window_end=launch_timestamp+post_launch_window_days*86400
                n_post,post_truncated,post_capped,post_pending=collect_time_window(
                    appid,launch_timestamp,post_window_end,review_file,seen_review_ids,
                    reviews_cap_per_window,'post_launch')

    # rows actually carrying the early access flag
    flagged_early_access=0
    with open(part_path(appid),encoding='utf-8') as review_file:
        for line in review_file:
            if '"written_during_early_access": true' in line or '"written_during_early_access":true' in line:
                flagged_early_access+=1

    record.update({'done':True,
                   'n_reviews':n_ea+n_post,
                   'n_flagged_early_access':flagged_early_access,
                   'n_ea_window':n_ea,
                   'n_post_window':n_post,
                   'truncated_windows':ea_truncated+post_truncated,
                   'cap_was_hit':bool(ea_capped or post_capped),
                   'windows_abandoned_at_cap':ea_pending+post_pending,
                   'requests_used':request_counter['total']-requests_before,
                   'seconds_taken':round(time.time()-started_at,1),
                   'collected_at':dt.datetime.now().isoformat(timespec='seconds')})

    # no early access rows means it fails inclusion, so mark it and keep the rows
    if flagged_early_access==0 and not still_in_early_access:
        record['excluded']='never_in_early_access'

    record['corpus_snapshot_id']=corpus_snapshot_id

    # checkpoint first, then swap the file in - os.replace is atomic
    checkpoint_file.write_text(json.dumps(record,indent=2))
    os.replace(part_path(appid),reviews_path(appid))

    # guard against a NameError here killing the whole worker pool
    print(f'app {appid}: {record["n_reviews"]} reviews '
          f'({flagged_early_access} flagged EA, cohort={record["cohort"]}, '
          f'{record["collection_strategy"]}) '
          f'in {record["requests_used"]} requests, {record["seconds_taken"]}s')
    return record


print('game collector is ready')

corpus snapshot id for this run: 20260805T005523
game collector is ready


# The twenty game spike

A small run to test the design before committing to the full sweep.

In [22]:
# the full sweep over the candidate list from steam_catalogue.ipynb
candidates=pd.read_csv(project_root/'data'/'processed'/'catalogue'/'candidate_games.csv')
spike_appids=candidates['appid'].tolist()

print('candidate games from the catalogue:',len(spike_appids))
print(candidates['cohort'].value_counts().to_string())
print('reviews available across candidates: {:,}'.format(candidates['total_reviews'].sum()))

print('collecting',len(spike_appids),'games using',parallel_workers,'workers')

spike_records=[]
spike_started=time.time()

# one game per worker, checkpointed and resumable
with ThreadPoolExecutor(max_workers=parallel_workers) as pool:
    submitted={pool.submit(collect_game,appid):appid for appid in spike_appids}
    for finished in as_completed(submitted):
        appid=submitted[finished]
        # one failure must not stop the run, so log it and carry on
        try:
            record=finished.result()
            if record:
                spike_records.append(record)
        except Exception as problem:
            print(f'app {appid}: FAILED with {type(problem).__name__}: {problem}')

print(f'spike finished in {(time.time()-spike_started)/60:.1f} minutes')
print(f'total requests: {request_counter["total"]}, retries: {request_counter["retries"]}, failures: {request_counter["failures"]}')

candidate games from the catalogue: 1887
cohort
launched       1220
still_in_ea     667
reviews available across candidates: 27,747,533
collecting 1887 games using 8 workers
app 739630: already done, 2100 reviews, skipping
app 1086940: already done, 6299 reviews, skipping
app 346110: already done, 6999 reviews, skipping
app 242760: already done, 8078 reviews, skipping
app 304930: already done, 7797 reviews, skipping
app 322330: already done, 6529 reviews, skipping
app 1966720: already done, 1000 reviews, skipping
app 221100: already done, 7590 reviews, skipping
app 892970: already done, 1900 reviews, skipping
app 1623730: already done, 0 reviews, skipping
app 291550: already done, 7640 reviews, skipping
app 251570: already done, 9300 reviews, skipping
app 548430: already done, 6400 reviews, skipping
app 632360: already done, 5800 reviews, skipping
app 3241660: already done, 500 reviews, skipping
app 1144200: already done, 5900 reviews, skipping
app 444090: already done, 6299 reviews, s

# Verifying every assumption i am about to build on

In [23]:
all_spike_records=pd.DataFrame(spike_records)
if 'excluded' not in all_spike_records.columns:
    all_spike_records['excluded']=None

# excluded games were never collected, reported separately
excluded_games=all_spike_records[all_spike_records['excluded'].notna()]
spike_summary=all_spike_records[all_spike_records['excluded'].isna()].copy()
spike_summary['ea_share']=(spike_summary['n_flagged_early_access']/spike_summary['n_reviews']).round(3)

print('GAMES EXCLUDED BEFORE COLLECTION')
if len(excluded_games):
    print(excluded_games[['appid','excluded']].to_string(index=False))
else:
    print('none')

display_columns=['appid','release_date','total_reviews','n_reviews','n_flagged_early_access',
                 'ea_share','requests_used','seconds_taken']

print('WHAT THE SPIKE ACTUALLY COLLECTED')
print(spike_summary[display_columns].to_string(index=False))

print(f'games collected            : {len(spike_summary)}')
print(f'total reviews on disk      : {spike_summary["n_reviews"].sum():,}')
print(f'total flagged early access : {spike_summary["n_flagged_early_access"].sum():,}')
print(f'total requests used        : {spike_summary["requests_used"].sum():,}')
print(f'reviews per request        : {spike_summary["n_reviews"].sum()/max(spike_summary["requests_used"].sum(),1):.1f}')
print(f'truncated windows          : {spike_summary["truncated_windows"].sum()}')

GAMES EXCLUDED BEFORE COLLECTION
  appid              excluded
1623730 released_too_recently
 387990 released_too_recently
  16900 released_too_recently
2488620 never_in_early_access
 918570 never_in_early_access
2904000 released_too_recently
2445690 released_too_recently
1849250 never_in_early_access
 491540 released_too_recently
2015270 released_too_recently
1001270 released_too_recently
 859580 never_in_early_access
1699480 released_too_recently
 236510 never_in_early_access
2307350 released_too_recently
3062930 never_in_early_access
 225600 never_in_early_access
2827230 released_too_recently
1648390 released_too_recently
 367270 released_too_recently
 242880 never_in_early_access
 976800 released_too_recently
1451480 released_too_recently
2510960 released_too_recently
1614550 released_too_recently
1863430 released_too_recently
2195120 released_too_recently
2570210 released_too_recently
2211170 released_too_recently
4233550 released_too_recently
1450900 released_too_recently
 708720

In [24]:
# load the rows back and check the fields per game
print('FIELD COVERAGE CHECK - loading the reviews back from disk')

coverage_rows=[]
for appid in spike_summary['appid']:
    review_rows=[json.loads(line) for line in open(reviews_path(appid),encoding='utf-8')]
    if not review_rows:
        continue
    reviews_frame=pd.json_normalize(review_rows)

    number_of_reviews=len(reviews_frame)
    has_ea_flag=reviews_frame['written_during_early_access'].notna().sum()
    has_playtime_at_review=(reviews_frame['author.playtime_at_review'].fillna(0)>0).sum()
    has_playtime_forever=(reviews_frame['author.playtime_forever'].fillna(0)>0).sum()
    has_steamid=reviews_frame['author.steamid'].notna().sum()

    # my metric: playtime after the review was written
    persistence=reviews_frame['author.playtime_forever']-reviews_frame['author.playtime_at_review']
    negative_persistence=(persistence<0).sum()

    coverage_rows.append({'appid':appid,
                          'reviews':number_of_reviews,
                          'ea_flag_%':round(100*has_ea_flag/number_of_reviews,1),
                          'playtime_at_review_%':round(100*has_playtime_at_review/number_of_reviews,1),
                          'playtime_forever_%':round(100*has_playtime_forever/number_of_reviews,1),
                          'steamid_%':round(100*has_steamid/number_of_reviews,1),
                          'negative_persistence_%':round(100*negative_persistence/number_of_reviews,1),
                          'median_persistence_hrs':round(persistence.median()/60,1)})

coverage_summary=pd.DataFrame(coverage_rows)
print(coverage_summary.to_string(index=False))

FIELD COVERAGE CHECK - loading the reviews back from disk
  appid  reviews  ea_flag_%  playtime_at_review_%  playtime_forever_%  steamid_%  negative_persistence_%  median_persistence_hrs
 251570     9300      100.0                 100.0               100.0      100.0                     0.0                    78.0
 892970     1900      100.0                 100.0               100.0      100.0                     0.0                    37.7
 304930     7797      100.0                  99.9               100.0      100.0                     0.0                    44.7
1086940     6299      100.0                 100.0               100.0      100.0                     0.0                   137.4
 291550     7640      100.0                 100.0               100.0      100.0                     0.0                    35.2
 221100     7590      100.0                  99.9               100.0      100.0                     0.0                    85.5
 346110     6999      100.0            

## The four assertions

In [25]:
print('my four assumptions')

# assumption 1 : the early access flag is present on essentially every review
worst_ea_flag_coverage=coverage_summary['ea_flag_%'].min()
print(f'1. written_during_early_access present on every review')
print(f'   worst game coverage: {worst_ea_flag_coverage}%')
assert worst_ea_flag_coverage>99, 'the early access flag is missing on some reviews, the cohort split needs rethinking'

# assumption 2 : the playtime fields are actually populated
worst_playtime_coverage=coverage_summary['playtime_forever_%'].min()
print(f'2. playtime_forever populated')
print(f'   worst game coverage: {worst_playtime_coverage}%')
assert worst_playtime_coverage>90, 'playtime is too sparse, post review persistence will not work'

# assumption 3 : post review persistence is computable on most games
games_with_persistence=(coverage_summary['median_persistence_hrs']>0).sum()
print(f'3. post review persistence computable (median above zero)')
print(f'   games where it works: {games_with_persistence} of {len(coverage_summary)}')
assert games_with_persistence>=15, 'persistence does not compute on enough games, this is my headline metric'

# assumption 4 : enough games clear the 200 review threshold
games_over_threshold=(spike_summary['n_flagged_early_access']>=200).sum()
print(f'4. games clearing the 200 early access review threshold')
print(f'   games above threshold: {games_over_threshold} of {len(spike_summary)}')

print('ALL FOUR ASSUMPTIONS HOLD - the design survives, i can build on this')

my four assumptions
1. written_during_early_access present on every review
   worst game coverage: 100.0%
2. playtime_forever populated
   worst game coverage: 99.6%
3. post review persistence computable (median above zero)
   games where it works: 1469 of 1701
4. games clearing the 200 early access review threshold
   games above threshold: 1318 of 1701
ALL FOUR ASSUMPTIONS HOLD - the design survives, i can build on this


# Sizing the full catalogue run

In [26]:
requests_per_game=spike_summary['requests_used'].mean()
seconds_per_game=spike_summary['seconds_taken'].mean()
reviews_per_game=spike_summary['n_reviews'].mean()

print('WHAT THE FULL RUN WILL COST')
print(f'average requests per game : {requests_per_game:.0f}')
print(f'average seconds per game  : {seconds_per_game:.1f}')
print(f'average reviews per game  : {reviews_per_game:,.0f}')

print('projected totals at this rate, running one game at a time')
for target_games in (500,1000,2000,3000):
    projected_hours=target_games*seconds_per_game/3600
    projected_reviews=target_games*reviews_per_game
    print(f'  {target_games:>5} games -> {projected_hours:>6.1f} hours, {projected_reviews:>12,.0f} reviews')

print('note: these games are deliberately the big popular ones so this is a pessimistic estimate.')
print('the median steam early access game is far smaller than rust or pubg.')
print('running several games in parallel will cut the wall clock further, sized in the catalogue notebook.')

WHAT THE FULL RUN WILL COST
average requests per game : 349
average seconds per game  : 35.4
average reviews per game  : 1,947
projected totals at this rate, running one game at a time
    500 games ->    4.9 hours,      973,388 reviews
   1000 games ->    9.8 hours,    1,946,777 reviews
   2000 games ->   19.7 hours,    3,893,553 reviews
   3000 games ->   29.5 hours,    5,840,330 reviews
note: these games are deliberately the big popular ones so this is a pessimistic estimate.
the median steam early access game is far smaller than rust or pubg.
running several games in parallel will cut the wall clock further, sized in the catalogue notebook.
